# Colab runner: smoke test, then the pilot

Before running: **Runtime > Change runtime type > GPU** (L4 or A100 if you have them; T4 works),
and put your Kaggle token as `aml-gnn-transfer/kaggle.json` in your Google Drive.
Colab does not run in the background: keep the tab open. If it disconnects, run all cells again:
finished runs have records on Drive and are skipped, an interrupted run's files are moved aside
(never deleted), and the data, prepared graphs and models are reused from Drive.

Every step stops the notebook if it fails. This file is written by `python -m tools.make_notebooks`.

`RUN_FULL = False` (default) runs the smoke test: source, then 10% scratch, then 10% fine-tune,
each capped at 2 epochs, then checks that scratch and finetune used the same labels and steps.
`RUN_FULL = True` runs the **pilot** (GIN only, one HI-Small source model, fractions 10/25/100%,
seeds 0 and 1, 3-epoch step budget). Turn it on only after the owner has approved the estimate
printed below (AGENTS.md rule 6).

In [ ]:
import os, shlex, subprocess, sys, time


def run(command):
    """Run a shell command, show its output as it arrives, and stop the notebook if it fails."""
    process = subprocess.Popen(
        ["bash", "-o", "pipefail", "-c", command],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
    )
    for line in process.stdout:
        print(line, end="", flush=True)
    code = process.wait()
    if code != 0:
        raise RuntimeError(f"Command failed (exit code {code}): {command}")


REPO_URL = "https://github.com/coachuttam/aml-gnn-transfer.git"

RUN_FULL = False  # False = smoke test only; True = the pilot (about 11 T4-hours: see the estimate cell)
WORK = "/content"
REPO = f"{WORK}/repo"

from google.colab import drive
drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/aml-gnn-transfer"  # kaggle.json lives here; everything we keep is saved here
os.makedirs(DRIVE, exist_ok=True)

# Data, prepared graphs, models and records all live on Drive, so a disconnect loses nothing.
# The pilot's records and models are kept apart from the full design's (and from smoke runs').
SMOKE_STAMP = time.strftime("%Y%m%d-%H%M%S")
RESULTS_HOME = f"{DRIVE}/pilot" if RUN_FULL else f"{DRIVE}/smoke/{SMOKE_STAMP}"
os.environ["AML_DATA_ROOT"] = f"{DRIVE}/data"
os.environ["AML_GRAPH_CACHE"] = f"{DRIVE}/graph_cache"
os.environ["AML_MODEL_ROOT"] = f"{RESULTS_HOME}/models"
os.environ["AML_RUN_RECORDS_DIR"] = f"{RESULTS_HOME}/records"
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"
print("Records and models go to:", RESULTS_HOME)

# Stop here if the GPU accelerator is off; remember which GPU we got.
run("nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv")
GPU_NAME = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], text=True
).splitlines()[0].strip()
print("GPU:", GPU_NAME)

In [ ]:
# Always run the latest pushed code: update an existing checkout instead of keeping a stale one.
if os.path.isdir(REPO):
    run(f"git -C {REPO} fetch origin && git -C {REPO} reset --hard origin/main")
else:
    run(f"git clone {REPO_URL} {REPO}")
run(f"git -C {REPO} submodule update --init")
run(f"git -C {REPO} log -1 --oneline")
run(f"git -C {REPO} submodule status")
os.chdir(REPO)
sys.path.insert(0, REPO)

Environment. First choice is Multi-GNN's own `env.yml` (micromamba, forced to a CUDA build of torch),
plus pyg-lib (the fast neighbour sampler, which `env.yml` lacks) as a wheel matched to that torch.
Whatever we get must pass `src.check_environment`: CUDA works, the PyG extensions import and run,
pyg-lib is in use. If it does not, we fall back to the notebook's own CUDA torch with all PyG add-ons
installed as matching wheels (`src.pyg_wheels`). If that also fails the check, the notebook stops.

In [ ]:
MICROMAMBA = f"{WORK}/bin/micromamba"
ENV_PYTHON = f"{MICROMAMBA} run -r {WORK}/mamba -n multignn python"

try:
    run(f"curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj -C {WORK} bin/micromamba")
    run(f"{MICROMAMBA} create -y -r {WORK}/mamba -n multignn -f external/Multi-GNN/env.yml 'pytorch=*=*cuda11.8*'")
    run(f"{ENV_PYTHON} -m src.pyg_wheels --only pyg_lib")
    run(f"{ENV_PYTHON} -m src.check_environment")
    PYTHON, ENV_LABEL = ENV_PYTHON, "env.yml"
except RuntimeError as problem:
    print("\nThe env.yml environment is not usable:", problem)
    print("Falling back to this notebook's torch with matching PyG wheels.\n")
    run(f"{sys.executable} -m src.pyg_wheels")
    run(f"{sys.executable} -m src.check_environment")  # raises if still wrong
    PYTHON, ENV_LABEL = sys.executable, "pip-fallback"
ENVIRONMENT_LABEL = f"{ENV_LABEL} / {GPU_NAME}"
print("Using:", ENVIRONMENT_LABEL)

Download HI-Small and LI-Small from Kaggle (skipped if already on Drive) and format them with IBM's script (CPU only, slow the first time).

In [ ]:
# The Kaggle token is read from Drive and copied where the Kaggle CLI looks. It is never printed.
TOKEN = f"{DRIVE}/kaggle.json"
if not os.path.exists(TOKEN):
    raise FileNotFoundError(f"Put your Kaggle API token (kaggle.json) in {DRIVE}/ first")
os.makedirs("/root/.kaggle", exist_ok=True)
run(f"cp {TOKEN} /root/.kaggle/kaggle.json && chmod 600 /root/.kaggle/kaggle.json")
run(f"{sys.executable} -m pip install -q kaggle")
run(f"{sys.executable} -m src.download_data")  # skips files already on Drive
run(f"{PYTHON} -m src.prepare --run-id prepare_only")  # IBM's formatter, skipped if already formatted

The runs. The first run of each dataset builds its prepared graph (several minutes of CPU, GPU idle) and saves it to Drive; every later train and score step loads it.

In [ ]:
def run_timed(label, command):
    """Run one step and print how long it took and on which GPU."""
    start = time.time()
    run(command)
    print(f"[{label}] {(time.time() - start) / 60:.1f} min on {GPU_NAME}", flush=True)


def smoke_chain(python, environment_label, seed, budget, fraction):
    """Source, then scratch and finetune at `fraction`, each as a smoke run."""
    label = shlex.quote(environment_label)
    for mode in ("source", "scratch", "finetune"):
        share = 1.0 if mode == "source" else fraction
        print(f"\n===== wrapper smoke run: {mode}, fraction {share} =====")
        run_timed(
            f"smoke {mode}",
            f"{python} -m src.run_experiment --model gin --mode {mode} --fraction {share} "
            f"--seed {seed} --step-budget-epochs {budget} --environment {label} --smoke-test",
        )

if RUN_FULL:
    print("RUN_FULL is on: skipping the smoke runs")
else:
    smoke_chain(PYTHON, ENVIRONMENT_LABEL, 0, 1, 0.10)
    run(f"{sys.executable} -m src.check_smoke")

The pilot (only when `RUN_FULL` is on). Prints the estimate first, then one timed line per run.

In [ ]:
if not RUN_FULL:
    print("RUN_FULL is off: no pilot runs")
else:
    from src.config import MODEL_ROOT, RUN_RECORDS_DIR
    from src.pilot import pilot_runs, record_exists, set_aside_partial_run

    run(f"{sys.executable} -m src.estimate_pilot")
    runs = pilot_runs()
    finished = sum(record_exists(item, RUN_RECORDS_DIR) for item in runs)
    print(f"\n{finished} of {len(runs)} pilot runs already have a record; those are skipped.")

    for number, item in enumerate(runs, start=1):
        if record_exists(item, RUN_RECORDS_DIR):
            print(f"[{number}/{len(runs)}] {item.run_id}: record exists, skipping")
            continue
        moved = set_aside_partial_run(item, MODEL_ROOT)  # a dead attempt's files, kept
        if moved:
            print(f"[{number}/{len(runs)}] earlier attempt of {item.run_id} moved to {moved}")
        words = " ".join(item.command_words())
        run_timed(
            f"{number}/{len(runs)} {item.run_id}",
            f"{PYTHON} -m src.run_experiment {words} --environment {shlex.quote(ENVIRONMENT_LABEL)}",
        )

What the records say (our scored values). Copy the records folder into the repo's `results/pilot/` and commit.

In [ ]:
run(f"grep -H -E '\"(minority_f1|epochs_requested|gradient_steps|labels_kept|kept_labels_fingerprint)\"' {os.environ['AML_RUN_RECORDS_DIR']}/*.json || true")